# Bonus: Incident Runbook — Model Performance Degradation

> **This notebook is optional.** Work through it if your group finishes the main flow early,
> or use it as a reference during client engagements when a drift alert fires in production.

---

**Scenario:** Monday 6:15am. You receive an alert:
> `Churn Drift Alert — your_schema: psi_score = 0.31 (threshold: 0.20)`

This notebook walks through the structured response. In a real engagement, this would live
as a Confluence or Notion runbook. Here it's executable so you can see exactly what each
diagnostic step produces.

In [0]:
%run ../utils/config

## Step 1: Detect — What Fired?

First, understand the scope of the issue.

In [0]:
# Check recent model performance metrics
perf_table = f"`{catalog}`.`{schema}`.churn_inference_log_profile_metrics"

try:
    perf_df = spark.table(perf_table)
    display(
        perf_df
        .filter("column_name = ':table'")
        .select("window.start", "percent_distinct", "count", "accuracy_score", "f1_score")
        .orderBy("window.start", ascending=False)
        .limit(10)
    )
except Exception:
    print("Profile metrics table not yet available.")
    print("Run 03_monitoring_setup.ipynb first and wait for the refresh.")

## Step 2: Diagnose — Which Features Drifted?

Identify the root cause — which input features changed?

In [0]:
drift_table = f"`{catalog}`.`{schema}`.churn_inference_log_drift_metrics"

try:
    drift_df = spark.table(drift_table)
    display(
        drift_df
        .filter("chi_squared_test.statistic IS NOT NULL")
        .select(
            "column_name",
            drift_df["chi_squared_test"]["statistic"].alias("drift_score"),
            "drift_type",
            "window.start",
        )
        .orderBy("drift_score", ascending=False)
        .limit(15)
    )
    print("\nHighest PSI = highest drift. Investigate those columns first.")
except Exception:
    print("Drift metrics not yet available.")

## Step 3: Decide — Retrain or Rollback?

| Situation | Action |
|---|---|
| Data pipeline bug (null rate changed, values truncated) | Fix pipeline, do NOT retrain |
| Temporary anomaly (holiday, outage, single bad batch) | Rollback, investigate |
| Permanent business change (new pricing, new market) | Retrain on fresh data |
| Model is just aging (gradual drift over weeks) | Schedule retrain |

**For today's scenario:** The acquisition campaign is ongoing (permanent change). We retrain.

## Step 4a: Rollback Path

If you decide to rollback while investigating:

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    ServedModelInput, ServedModelInputWorkloadSize, TrafficConfig, Route
)
import mlflow
from mlflow import MlflowClient

w = WorkspaceClient()
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

endpoint_name = f"churn_{safe_username}"
model_name    = f"{catalog}.{schema}.churn_classifier"

# Find the previous champion (version before current)
versions = sorted(
    client.search_model_versions(f"name='{model_name}'"),
    key=lambda v: int(v.version)
)

if len(versions) < 2:
    print("Only one version exists — cannot rollback. Use the first version as a baseline.")
    rollback_version = versions[0].version
else:
    # Previous version = second most recent
    rollback_version = versions[-2].version

print(f"Rolling back to version: {rollback_version}")

w.serving_endpoints.update_config_and_wait(
    name=endpoint_name,
    served_models=[
        ServedModelInput(
            model_name=model_name,
            model_version=str(rollback_version),
            workload_size=ServedModelInputWorkloadSize.SMALL,
            scale_to_zero_enabled=True,
        )
    ],
    traffic_config=TrafficConfig(routes=[
        Route(served_model_name=f"{model_name}-{rollback_version}", traffic_percentage=100)
    ])
)
print(f"✓ Rolled back to version {rollback_version}")

## Step 4b: Retrain Path

If you decide retraining is the right call, trigger the CI/CD pipeline:

In [0]:
# Trigger the retrain job (same as 05_trigger_retrain.ipynb)
from databricks.sdk import WorkspaceClient
import time

w = WorkspaceClient()
jobs_list = list(w.jobs.list(name="workshop_retrain_churn"))
if jobs_list:
    job = jobs_list[0]
    run_response = w.jobs.run_now(job_id=job.job_id)
    print(f"✓ Retrain job triggered. Run ID: {run_response.run_id}")
    print(f"  Monitor at: {w.config.host}#job/{job.job_id}/run/{run_response.run_id}")
else:
    print("Retrain job not found. Trigger manually via the Workflows UI.")

## Step 5: Verify

After rollback or retraining completes:
1. Trigger another monitoring refresh
2. Check that the new performance metrics are within acceptable range
3. Close the incident

In [0]:
# Trigger monitoring refresh to check current state
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

full_table_name = f"{catalog}.{schema}.churn_inference_log"
try:
    refresh = w.quality_monitors.run_refresh(table_name=full_table_name)
    print(f"✓ Monitoring refresh triggered. Refresh ID: {refresh.refresh_id}")
    print("Wait 2-3 minutes, then re-run Step 1 to verify metrics improved.")
except Exception as e:
    print(f"Could not trigger refresh: {e}")

## Step 6: Post-Incident Documentation

Always document incidents — especially for client handover.

**Incident report template:**

```
Date: YYYY-MM-DD
Alert: Churn model PSI > 0.2 on feature: tenure
Severity: Medium (model degradation, not outage)

Timeline:
  06:15  Alert fired
  06:30  On-call engineer diagnosed: tenure distribution shift
  07:00  Root cause: acquisition campaign changed customer cohort
  07:15  Decision: retrain (permanent business change)
  09:30  Retrain completed, new champion deployed
  10:00  Monitoring refresh confirmed metrics restored

Root cause: Input distribution shift due to business change
Action taken: Full retrain on updated data
Prevention: Add tenure distribution check to data ingestion job
```

## Final Discussion: Operability & Client Handover

- **Who should own this runbook when you leave the engagement?**
- **What access do client engineers need?** (UC permissions, job trigger rights, alert subscriptions)
- **What does 'done' look like for this engagement?**
  - Model deployed ✅
  - Monitoring configured ✅
  - Alerts set up ✅
  - Runbook documented ✅
  - Client team trained ✅
  - On-call rotation defined ✅